### Choisir et importer un algorithme de ML (Algorithme de Machine Learning)

In [3]:
# Les Bibliothèques
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE


In [12]:
# importons notre df_features
df_features = pd.read_csv("../data/processed/df_features.csv")

display(df_features.head())



,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,senior,high_glucose,cardio_risk
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,1,1,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1,0,1,0
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,1,0,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,0,1,0
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,1,1,1


##### Nous allons faire une rapide vérification sécuritaire

In [13]:
print(df_features.shape)

(5109, 14)


In [15]:
print(df_features.info())

<class 'pandas.DataFrame'>
RangeIndex: 5109 entries, 0 to 5108
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5109 non-null   str    
 1   age                5109 non-null   float64
 2   hypertension       5109 non-null   int64  
 3   heart_disease      5109 non-null   int64  
 4   ever_married       5109 non-null   str    
 5   work_type          5109 non-null   str    
 6   Residence_type     5109 non-null   str    
 7   avg_glucose_level  5109 non-null   float64
 8   bmi                5109 non-null   float64
 9   smoking_status     5109 non-null   str    
 10  stroke             5109 non-null   int64  
 11  senior             5109 non-null   int64  
 12  high_glucose       5109 non-null   int64  
 13  cardio_risk        5109 non-null   int64  
dtypes: float64(3), int64(6), str(5)
memory usage: 558.9 KB
None


In [11]:
print(df_features.isna().sum())

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
senior               0
high_glucose         0
cardio_risk          0
dtype: int64


In [14]:
print(df_features.duplicated().sum())

0


##### Nous alors refaire le preprocessing avec ce dataset de manière plus fluide

In [16]:
# définissons nos variables
X = df_features.drop(columns = "stroke")
# notre variable cible
y = df_features["stroke"]


In [17]:
# vérifions
X.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,senior,high_glucose,cardio_risk
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,1,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,0,1,0
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,0,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,0,1,0
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,1,1


In [20]:
y.info()

<class 'pandas.Series'>
RangeIndex: 5109 entries, 0 to 5108
Series name: stroke
Non-Null Count  Dtype
--------------  -----
5109 non-null   int64
dtypes: int64(1)
memory usage: 40.0 KB


In [21]:
# identifions les variables
cat_features = X.select_dtypes(include = str).columns

num_features = X.select_dtypes(exclude = str).columns


In [25]:
# Vérifions
print("cat_features contient: " , cat_features)

print()

print("num_features contient: " , num_features)

cat_features contient:  Index(['gender', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='str')

num_features contient:  Index(['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi',
       'senior', 'high_glucose', 'cardio_risk'],
      dtype='str')


In [26]:
# Séparons nos variables par le Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
                                    X,
                                    y,
                                    random_state=42,
                                    test_size= 0.20,
                                    stratify=y

                                        )


In [28]:
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)


X_train:  (4087, 13)
X_test:  (1022, 13)
y_train:  (4087,)
y_test:  (1022,)


In [29]:
# Vérification des valeurs manquantes comme nous l'avons fait ci-dessus
print(df_features.isna().sum())

# Aucune valeur manquante.
# L'imputation a déjà été réalisée lors de la création du dataset clean.

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
senior               0
high_glucose         0
cardio_risk          0
dtype: int64


#### Préparation finale des données avant l'entraînement des modèles

###### Encodage des variables catégorielles

In [43]:
# nos variables catégorielles
print(cat_features)

Index(['gender', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='str')


In [ ]:
# modèle encodeur
encoder = OneHotEncoder(drop= 'first',
                        handle_unknown='ignore',
                        sparse_output= False)


In [31]:
# Apprend et transforme nos variables train catégorielles
X_train_encoded_array = encoder.fit_transform(X_train[cat_features])

# Transforme nos variables test catégorielles 
X_test_encoded_array = encoder.transform(X_test[cat_features])

# Récupérons les noms des colonnes 
encoded_columns = encoder.get_feature_names_out(cat_features)


In [37]:
# Construisons les Dataframes
# Dataframe Train
X_train_encoded = pd.DataFrame(
                            X_train_encoded_array,
                            columns= encoded_columns,
                            index= X_train.index
                            )

X_train_encoded.head()


,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3744,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4183,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3409,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
284,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
# Dataframe Test
X_test_encoded = pd.DataFrame(
                            X_test_encoded_array,
                            columns= encoded_columns,
                            index= X_test.index
                                )

X_test_encoded.head()


,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
3666,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2217,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
374,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2392,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
299,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
print("X_train_encoded:" , X_train_encoded.shape)
print("X_test_encoded: ", X_test_encoded.shape)

X_train_encoded: (4087, 10)
X_test_encoded:  (1022, 10)


In [ ]:
# vérifions qu'il y'a pas de nan après encodage
X_train_encoded.isna().sum()

gender_Male                       0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

In [42]:
X_test_encoded.isna().sum()

gender_Male                       0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

##### Standardisation des variables numériques

In [44]:
# Variables numériques
print(num_features)

Index(['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi',
       'senior', 'high_glucose', 'cardio_risk'],
      dtype='str')


In [45]:
# Nous observons que parmis nos variables numériques, certaines sont déjà numériques et binaires donc n'ont 
# plus besoin d'être standardiser.
# Afin que l'ensemble de nos variable snumériques soient sur la même échelle, nous allons standardiser les variables qui en ont besoin
# telles que: age, avg_glucose_level et bmi

X_train[num_features].head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,senior,high_glucose,cardio_risk
845,48.0,0,0,69.21,33.1,0,0,0
3744,29.0,0,0,84.19,21.2,0,0,0
4183,35.0,0,0,119.40,22.9,0,0,0
3409,38.0,0,0,108.68,32.7,0,0,0
284,14.0,0,0,82.34,31.6,0,0,0


In [ ]:
# Nous allons classer nos variables numériques continues
continuous_features = [
    "age",
    "avg_glucose_level",
    "bmi"
]

# Classons nos variables numériques binaires
binary_features = [
    "hypertension",
    "heart_disease",
    "senior",
    "high_glucose",
    "cardio_risk"
]


In [49]:
# modèle scaler
scaler = StandardScaler()


In [50]:
# Stantardissons uniquement nos variables numériques continues
# Jeu d'entrainement
X_train[continuous_features] = scaler.fit_transform(X_train[continuous_features])

# Jeu de test
X_test[continuous_features] = scaler.transform(X_test[continuous_features])


In [51]:
# Vérification
X_train[continuous_features].describe().round(2)

,age,avg_glucose_level,bmi
count,4087.00,4087.00,4087.00
mean,-0.00,0.00,-0.00
std,1.00,1.00,1.00
min,-1.91,-1.13,-2.40
25%,-0.81,-0.64,-0.66
50%,0.08,-0.31,-0.10
75%,0.78,0.18,0.51
max,1.71,3.71,8.89
